In [0]:
%run /Users/mathisneha2004@gmail.com/config/Pipeline_Config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, StringType, DateType

print("✓ Libraries imported successfully")

In [0]:
from datetime import datetime
silver_start = datetime.now()
silver_input = spark.table(BRONZE_CLAIMS_TABLE).count()

print(f"{'='*50}")
print(f"SILVER LAYER AUDIT LOG")
print(f"{'='*50}")
print(f"Start Time    : {silver_start}")
print(f"Input Records : {silver_input:,}")
print(f"{'='*50}")

In [0]:
try:
    print("Reading Bronze layer tables...\n")

    claims_bronze = spark.table(BRONZE_CLAIMS_TABLE)
    hospital_bronze = spark.table(BRONZE_HOSPITAL_TABLE)

    claims_count = claims_bronze.count()
    hospital_count = hospital_bronze.count()

    print(f"✓ Claims Bronze: {claims_count:,} records")
    print(f"✓ Hospital Bronze: {hospital_count:,} records")
    print(f"\nStarting Silver Layer transformations...\n")
except Exception as e:
    print(f"✗ Failed to read Bronze tables: {str(e)}")
    raise

   
## T1 - Fix Date Columns and Rename Patient ID
* Overwrite SP_STATE_CODE with zero-padded 2-character string
* Rename DESYNPUF_ID to PATIENT_ID for clarity

In [0]:
try:
    print("T1: Converting SP_STATE_CODE to zero-padded format and renaming patient ID...")

    claims_t1 = claims_bronze.withColumn(
        "SP_STATE_CODE",
        F.lpad(F.col("SP_STATE_CODE").cast("string"), 2, "0")
    ).withColumnRenamed(
        "DESYNPUF_ID", "PATIENT_ID"
    )

    print("✓ T1 Complete: SP_STATE_CODE converted, PATIENT_ID renamed")
except Exception as e:
    print(f"✗ T1 Failed: {str(e)}")
    raise

   
## T2 - Decode Gender and Race
Overwrite columns: BENE_SEX_IDENT_CD, BENE_RACE_CD, BENE_ESRD_IND with decoded values

In [0]:
try:
    print("T2: Decoding gender, race, and kidney failure...")

    claims_t2 = claims_t1.withColumn(
        "BENE_SEX_IDENT_CD",
        F.when(F.col("BENE_SEX_IDENT_CD") == 1, "Male")
         .when(F.col("BENE_SEX_IDENT_CD") == 2, "Female")
         .otherwise("Unknown")
    ).withColumn(
        "BENE_RACE_CD",
        F.when(F.col("BENE_RACE_CD") == 1, "White")
         .when(F.col("BENE_RACE_CD") == 2, "Black")
         .when(F.col("BENE_RACE_CD") == 3, "Other")
         .when(F.col("BENE_RACE_CD") == 4, "Asian")
         .when(F.col("BENE_RACE_CD") == 5, "Hispanic")
         .when(F.col("BENE_RACE_CD") == 6, "American Indian")
         .otherwise("Unknown")
    ).withColumn(
        "BENE_ESRD_IND",
        F.when(F.col("BENE_ESRD_IND") == "Y", "Yes")
         .when(F.col("BENE_ESRD_IND") == "N", "No")
         .otherwise("Unknown")
    )

    print(f"✓ T2 Complete: BENE_SEX_IDENT_CD, BENE_RACE_CD, BENE_ESRD_IND decoded")
except Exception as e:
    print(f"✗ T2 Failed: {str(e)}")
    raise

   
## T3 - Decode Disease Flags
Overwrite 11 SP_ columns with decoded values (Yes/No/Unknown)

In [0]:
try:
    print("T3: Decoding 11 chronic disease flags...")

    def decode_disease(col_name):
        return F.when(F.col(col_name) == 1, "Yes")\
                .when(F.col(col_name) == 2, "No")\
                .otherwise("Unknown")

    claims_t3 = claims_t2.withColumn(
        "SP_ALZHDMTA", decode_disease("SP_ALZHDMTA")
    ).withColumn(
        "SP_CHF", decode_disease("SP_CHF")
    ).withColumn(
        "SP_CHRNKIDN", decode_disease("SP_CHRNKIDN")
    ).withColumn(
        "SP_CNCR", decode_disease("SP_CNCR")
    ).withColumn(
        "SP_COPD", decode_disease("SP_COPD")
    ).withColumn(
        "SP_DEPRESSN", decode_disease("SP_DEPRESSN")
    ).withColumn(
        "SP_DIABETES", decode_disease("SP_DIABETES")
    ).withColumn(
        "SP_ISCHMCHT", decode_disease("SP_ISCHMCHT")
    ).withColumn(
        "SP_OSTEOPRS", decode_disease("SP_OSTEOPRS")
    ).withColumn(
        "SP_RA_OA", decode_disease("SP_RA_OA")
    ).withColumn(
        "SP_STRKETIA", decode_disease("SP_STRKETIA")
    )

    print(f"✓ T3 Complete: 11 SP_ disease columns decoded")
except Exception as e:
    print(f"✗ T3 Failed: {str(e)}")
    raise

## T4 - Handle Null Values
Add new _CLEAN columns with NULL values replaced

In [0]:
try:
    print("T4: Handling null values...")

    claims_t4 = claims_t3.withColumn(
        "LINE_NCH_PMT_AMT_1_CLEAN",
        F.coalesce(F.col("LINE_NCH_PMT_AMT_1"), F.lit(0.00))
    ).withColumn(
        "LINE_BENE_PTB_DDCTBL_AMT_1_CLEAN",
        F.coalesce(F.col("LINE_BENE_PTB_DDCTBL_AMT_1"), F.lit(0.00))
    ).withColumn(
        "LINE_COINSRNC_AMT_1_CLEAN",
        F.coalesce(F.col("LINE_COINSRNC_AMT_1"), F.lit(0.00))
    ).withColumn(
        "PRF_PHYSN_NPI_1_CLEAN",
        F.when(F.col("PRF_PHYSN_NPI_1").isNull(), "UNKNOWN")
         .otherwise(F.col("PRF_PHYSN_NPI_1"))
    ).withColumn(
        "ICD9_DGNS_CD_1_CLEAN",
        F.when(F.col("ICD9_DGNS_CD_1").isNull(), "UNKNOWN")
         .otherwise(F.col("ICD9_DGNS_CD_1"))
    ).withColumn(
        "LINE_ICD9_DGNS_CD_1_CLEAN",
        F.when(F.col("LINE_ICD9_DGNS_CD_1").isNull(), "UNKNOWN")
         .otherwise(F.col("LINE_ICD9_DGNS_CD_1"))
    )

    print(f"✓ T4 Complete: 6 _CLEAN columns added")
except Exception as e:
    print(f"✗ T4 Failed: {str(e)}")
    raise

## T6 - Calculate Payment Totals
Add new column TOTAL_PAYMENT_AMOUNT

In [0]:
try:
    print("T6: Calculating total payment amounts...")

    claims_t6 = claims_t4.withColumn(
        "TOTAL_PAYMENT_AMOUNT",
        F.round(
            F.col("LINE_NCH_PMT_AMT_1_CLEAN") + 
            F.col("LINE_BENE_PTB_DDCTBL_AMT_1_CLEAN") + 
            F.col("LINE_COINSRNC_AMT_1_CLEAN"),
            2
        )
    )

    print(f"✓ T6 Complete: TOTAL_PAYMENT_AMOUNT calculated")
except Exception as e:
    print(f"✗ T6 Failed: {str(e)}")
    raise

## T7 - Calculate Age and Age Group
Add new columns: AGE, AGE_GROUP

In [0]:
try:
    print("T7: Calculating age and age groups...")

    def parse_date_int(col_name):
        date_str = F.col(col_name).cast("string")
        return F.when(
            (F.col(col_name).isNull()) | (F.col(col_name) == 0) | (F.length(date_str) != 8),
            F.lit(None).cast("date")
        ).otherwise(
            F.to_date(date_str, "yyyyMMdd")
        )

    claims_t7 = claims_t6.withColumn(
        "AGE",
        F.floor(
            F.datediff(
                parse_date_int("CLM_FROM_DT"),
                parse_date_int("BENE_BIRTH_DT")
            ) / 365.25
        ).cast(IntegerType())
    ).withColumn(
        "AGE_GROUP",
        F.when(F.col("AGE") < 18, "0-17")
         .when((F.col("AGE") >= 18) & (F.col("AGE") <= 44), "18-44")
         .when((F.col("AGE") >= 45) & (F.col("AGE") <= 64), "45-64")
         .when((F.col("AGE") >= 65) & (F.col("AGE") <= 74), "65-74")
         .when((F.col("AGE") >= 75) & (F.col("AGE") <= 84), "75-84")
         .when(F.col("AGE") >= 85, "85+")
         .otherwise("Unknown")
    )

    print(f"✓ T7 Complete: AGE and AGE_GROUP calculated")
except Exception as e:
    print(f"✗ T7 Failed: {str(e)}")
    raise

## T8 - Decode Claim Status
Add new columns: CLAIM_STATUS, CLAIM_TYPE

In [0]:
try:
    print("T8: Decoding claim status and type...")

    claims_t8 = claims_t7.withColumn(
        "CLAIM_STATUS",
        F.when(F.col("LINE_PRCSG_IND_CD_1") == "A", "Assigned")
         .when(F.col("LINE_PRCSG_IND_CD_1") == "N", "Non-Assigned")
         .when(F.col("LINE_PRCSG_IND_CD_1") == "R", "Rejected")
         .otherwise("Unknown")
    ).withColumn(
        "CLAIM_TYPE",
        F.when(F.col("HCPCS_CD_1").between("99201", "99499"), "Office or Outpatient Visit")
         .when(F.col("HCPCS_CD_1").between("70000", "79999"), "Radiology")
         .when(F.col("HCPCS_CD_1").between("80000", "89999"), "Laboratory")
         .when(F.col("HCPCS_CD_1").between("90000", "99000"), "Medicine or Procedure")
         .when(F.col("HCPCS_CD_1").between("10000", "69999"), "Surgery")
         .otherwise("Other")
    )

    print(f"✓ T8 Complete: CLAIM_STATUS and CLAIM_TYPE added")
except Exception as e:
    print(f"✗ T8 Failed: {str(e)}")
    raise

## T9 - Disease Count and Risk Category
Add new columns: DISEASE_COUNT, RISK_CATEGORY

In [0]:
try:
    print("T9: Calculating disease count and risk category...")

    disease_columns = [
        "SP_ALZHDMTA", "SP_CHF", "SP_CHRNKIDN", "SP_CNCR", 
        "SP_COPD", "SP_DEPRESSN", "SP_DIABETES", "SP_ISCHMCHT", 
        "SP_OSTEOPRS", "SP_RA_OA", "SP_STRKETIA"
    ]

    disease_count_expr = sum(
        F.when(F.col(col) == "Yes", 1).otherwise(0) for col in disease_columns
    )

    claims_t9 = claims_t8.withColumn(
        "DISEASE_COUNT",
        disease_count_expr
    ).withColumn(
        "RISK_CATEGORY",
        F.when(F.col("DISEASE_COUNT") == 0, "Low Risk")
         .when(F.col("DISEASE_COUNT").between(1, 2), "Moderate Risk")
         .when(F.col("DISEASE_COUNT").between(3, 4), "High Risk")
         .when(F.col("DISEASE_COUNT") >= 5, "Very High Risk")
         .otherwise("Unknown")
    )

    print(f"✓ T9 Complete: DISEASE_COUNT and RISK_CATEGORY calculated")
except Exception as e:
    print(f"✗ T9 Failed: {str(e)}")
    raise

## T10 - Claim Duration and Type
Add new columns: CLAIM_DURATION_DAYS, CLAIM_DURATION_TYPE

In [0]:
try:
    print("T10: Calculating claim duration and type...")

    def parse_date_int(col_name):
        date_str = F.col(col_name).cast("string")
        return F.when(
            (F.col(col_name).isNull()) | (F.col(col_name) == 0) | (F.length(date_str) != 8),
            F.lit(None).cast("date")
        ).otherwise(
            F.to_date(date_str, "yyyyMMdd")
        )

    claims_t10 = claims_t9.withColumn(
        "CLAIM_DURATION_DAYS",
        F.datediff(
            parse_date_int("CLM_THRU_DT"),
            parse_date_int("CLM_FROM_DT")
        ).cast(IntegerType())
    ).withColumn(
        "CLAIM_DURATION_TYPE",
        F.when(F.col("CLAIM_DURATION_DAYS") == 0, "Same-day")
         .when(F.col("CLAIM_DURATION_DAYS").between(1, 7), "Short Stay (1-7 days)")
         .when(F.col("CLAIM_DURATION_DAYS").between(8, 30), "Medium Stay (8-30 days)")
         .when(F.col("CLAIM_DURATION_DAYS") > 30, "Long Stay (30+ days)")
         .otherwise("Invalid")
    )

    print(f"✓ T10 Complete: CLAIM_DURATION_DAYS and CLAIM_DURATION_TYPE calculated")
except Exception as e:
    print(f"✗ T10 Failed: {str(e)}")
    raise

## T11 - State Benchmark Enrichment
Add new columns: STATE_AVG_PAYMENT, STATE_BENCHMARK_CATEGORY

In [0]:
try:
    print("T11: Calculating state benchmarks...")

    # Rename provider organization column and prepare hospital data
    hospital_state = hospital_bronze.withColumnRenamed(
        "Rndrng_Prvdr_Org_Name", "HCO"
    ).withColumn(
        "STATE_FIPS_PADDED",
        F.lpad(F.col("Rndrng_Prvdr_State_FIPS").cast("string"), 2, "0")
    )

    state_avg_payments = hospital_state.groupBy("STATE_FIPS_PADDED").agg(
        F.avg("Avg_Mdcr_Pymt_Amt").alias("STATE_AVG_MEDICARE_PAYMENT")
    )

    national_avg = state_avg_payments.select(F.avg("STATE_AVG_MEDICARE_PAYMENT")).first()[0]
    print(f"  National average payment: ${national_avg:,.2f}")

    claims_t11 = claims_t10.join(
        state_avg_payments,
        claims_t10.SP_STATE_CODE == state_avg_payments.STATE_FIPS_PADDED,
        "left"
    ).withColumn(
        "STATE_BENCHMARK_CATEGORY",
        F.when(F.col("STATE_AVG_MEDICARE_PAYMENT").isNull(), "No Data")
         .when(F.col("STATE_AVG_MEDICARE_PAYMENT") > national_avg * 1.05, "Above National Average")
         .when(F.col("STATE_AVG_MEDICARE_PAYMENT") < national_avg * 0.95, "Below National Average")
         .otherwise("At National Average")
    ).drop("STATE_FIPS_PADDED")

    print(f"✓ T11 Complete: STATE_AVG_MEDICARE_PAYMENT and STATE_BENCHMARK_CATEGORY added")
except Exception as e:
    print(f"✗ T11 Failed: {str(e)}")
    raise

## T12 - Data Validation
Add new columns: DATA_QUALITY_FLAG, VALIDATION_NOTES

In [0]:
try:
    print("T12: Applying data validation rules...")

    def parse_date_int(col_name):
        date_str = F.col(col_name).cast("string")
        return F.when(
            (F.col(col_name).isNull()) | (F.col(col_name) == 0) | (F.length(date_str) != 8),
            F.lit(None).cast("date")
        ).otherwise(
            F.to_date(date_str, "yyyyMMdd")
        )

    validation_conditions = [
        (
            (parse_date_int("BENE_DEATH_DT").isNotNull()) & 
            (F.col("BENE_DEATH_DT") != 0) & 
            (parse_date_int("CLM_FROM_DT") > parse_date_int("BENE_DEATH_DT")),
            "Claim after death"
        ),
        (F.col("TOTAL_PAYMENT_AMOUNT") < 0, "Negative payment"),
        (parse_date_int("CLM_FROM_DT") > parse_date_int("CLM_THRU_DT"), "Start after end date"),
        ((F.col("AGE") < 0) | (F.col("AGE") > 120), "Invalid age")
    ]

    validation_notes_expr = F.concat_ws(
        " | ",
        *[F.when(condition, reason) for condition, reason in validation_conditions]
    )

    claims_t12 = claims_t11.withColumn(
        "VALIDATION_NOTES",
        validation_notes_expr
    ).withColumn(
        "DATA_QUALITY_FLAG",
        F.when(
            (F.col("VALIDATION_NOTES").isNull()) | (F.col("VALIDATION_NOTES") == ""),
            DATA_QUALITY_PASS
        ).otherwise("FAIL")
    )

    fail_count = claims_t12.filter(F.col("DATA_QUALITY_FLAG") == "FAIL").count()
    pass_count = claims_t12.filter(F.col("DATA_QUALITY_FLAG") == DATA_QUALITY_PASS).count()

    print(f"✓ T12 Complete: DATA_QUALITY_FLAG and VALIDATION_NOTES added")
    print(f"  Passed validation: {pass_count:,}")
    print(f"  Failed validation: {fail_count:,}")
except Exception as e:
    print(f"✗ T12 Failed: {str(e)}")
    raise

   
   
## T5 - Pass Through All Records
Applied LAST after all transformations.
* ALL records → Silver table (no duplicate tracking)
* No duplicate flagging or S3 audit logging

In [0]:
try:
    print("T5: Passing through all records without duplicate tracking...")

    initial_count = claims_t12.count()

    # Pass all records to Silver table without adding duplicate columns
    claims_silver_final = claims_t12

    print(f"✓ T5 Complete: All records passed through")
    print(f"  Total records going to Silver: {initial_count:,}")
    print(f"  No duplicate tracking applied")

    print(f"\nProceeding with ALL {initial_count:,} records to Silver table...")
except Exception as e:
    print(f"✗ T5 Failed: {str(e)}")
    raise

In [0]:
%skip
%sql
CREATE SCHEMA IF NOT EXISTS healthcare_claims_data.silver
COMMENT 'Silver layer - cleaned and enriched healthcare claims data with duplicate tracking'

In [0]:
try:
    print("Writing to Silver layer...\n")

    target_table = SILVER_CLAIMS_TABLE

    claims_silver_final.write.format("delta") \
        .mode(SILVER_WRITE_MODE) \
        .option("overwriteSchema", OVERWRITE_SCHEMA) \
        .saveAsTable(target_table)

    print(f"✓ Silver table written successfully: {target_table}")
    print(f"  Mode: overwrite")
    print(f"  Schema evolution: enabled\n")
except Exception as e:
    print(f"✗ Failed to write Silver table: {str(e)}")
    raise

In [0]:
try:
    silver_claims = spark.table(SILVER_CLAIMS_TABLE)

    total_records = silver_claims.count()
    print(f"Total Records: {total_records:,}\n")

    print("Sample of 10 records showing key new columns:\n")

    display_cols = [
        "PATIENT_ID", "CLM_ID",
        "SP_STATE_CODE", "BENE_SEX_IDENT_CD", "BENE_RACE_CD", "AGE", "AGE_GROUP",
        "DISEASE_COUNT", "RISK_CATEGORY", "TOTAL_PAYMENT_AMOUNT",
        "CLAIM_DURATION_DAYS", "CLAIM_DURATION_TYPE", "CLAIM_STATUS", "CLAIM_TYPE",
        "STATE_BENCHMARK_CATEGORY", "DATA_QUALITY_FLAG"
    ]

    display(silver_claims.select(display_cols).limit(10))
except Exception as e:
    print(f"✗ Verification failed: {str(e)}")
    raise

In [0]:
try:
    print("="*80)
    print("SILVER LAYER SUMMARY")
    print("="*80)

    quality_summary = silver_claims.groupBy("DATA_QUALITY_FLAG").count().orderBy("DATA_QUALITY_FLAG")
    print("\nData Quality Distribution:")
    quality_summary.show()

    risk_summary = silver_claims.groupBy("RISK_CATEGORY").count().orderBy("count", ascending=False)
    print("Risk Category Distribution:")
    risk_summary.show()

    age_summary = silver_claims.groupBy("AGE_GROUP").count().orderBy("AGE_GROUP")
    print("Age Group Distribution:")
    age_summary.show()

    print("="*80)
    print("✓ Silver Layer transformation pipeline complete!")
    print("="*80)
except Exception as e:
    print(f"✗ Summary statistics failed: {str(e)}")
    raise

---
# Hospital Silver Layer Transformations
Transform hospital bronze data to silver layer

## H1 - Add State FIPS Padded
Keep original column names, only add STATE_FIPS_PADDED

In [0]:
try:
    print("H1: Adding STATE_FIPS_PADDED column...")
    
    hospital_t1 = hospital_bronze.withColumn(
        "STATE_FIPS_PADDED",
        F.lpad(F.col("Rndrng_Prvdr_State_FIPS").cast("string"), 2, "0")
    )
    
    print("✓ H1 Complete: STATE_FIPS_PADDED column added")
except Exception as e:
    print(f"✗ H1 Failed: {str(e)}")
    raise

## H2 - Handle Null Values
Replace nulls in numeric fields with 0

In [0]:
try:
    print("H2: Handling null values in hospital data...")
    
    hospital_t2 = hospital_t1.withColumn(
        "Tot_Dschrgs",
        F.coalesce(F.col("Tot_Dschrgs"), F.lit(0))
    ).withColumn(
        "Avg_Submtd_Cvrd_Chrg",
        F.coalesce(F.col("Avg_Submtd_Cvrd_Chrg"), F.lit(0.0))
    ).withColumn(
        "Avg_Tot_Pymt_Amt",
        F.coalesce(F.col("Avg_Tot_Pymt_Amt"), F.lit(0.0))
    ).withColumn(
        "Avg_Mdcr_Pymt_Amt",
        F.coalesce(F.col("Avg_Mdcr_Pymt_Amt"), F.lit(0.0))
    )
    
    print("✓ H2 Complete: Null values handled for numeric columns")
except Exception as e:
    print(f"✗ H2 Failed: {str(e)}")
    raise

## H3 - Calculate Derived Metrics
Add calculated columns for analysis

In [0]:
try:
    print("H3: Calculating derived metrics...")
    
    hospital_t3 = hospital_t2.withColumn(
        "PAYMENT_TO_CHARGE_RATIO",
        F.when(
            F.col("Avg_Submtd_Cvrd_Chrg") > 0,
            F.round(F.col("Avg_Tot_Pymt_Amt") / F.col("Avg_Submtd_Cvrd_Chrg"), 4)
        ).otherwise(F.lit(None))
    ).withColumn(
        "MEDICARE_PAYMENT_PCT",
        F.when(
            F.col("Avg_Tot_Pymt_Amt") > 0,
            F.round((F.col("Avg_Mdcr_Pymt_Amt") / F.col("Avg_Tot_Pymt_Amt")) * 100, 2)
        ).otherwise(F.lit(None))
    ).withColumn(
        "LOCATION_TYPE",
        F.when(F.col("Rndrng_Prvdr_RUCA").isin(["1", "2", "3"]), "Urban")
         .when(F.col("Rndrng_Prvdr_RUCA").isin(["4", "5", "6"]), "Large Rural")
         .when(F.col("Rndrng_Prvdr_RUCA").isin(["7", "8", "9", "10"]), "Small Rural")
         .otherwise("Unknown")
    )
    
    print("✓ H3 Complete: Derived metrics calculated")
except Exception as e:
    print(f"✗ H3 Failed: {str(e)}")
    raise

## H4 - Data Validation
Add data quality flags and validation notes

In [0]:
try:
    print("H4: Applying data validation rules...")
    
    validation_conditions = [
        (F.col("Avg_Submtd_Cvrd_Chrg") < 0, "Negative submitted charge"),
        (F.col("Avg_Tot_Pymt_Amt") < 0, "Negative total payment"),
        (F.col("Avg_Mdcr_Pymt_Amt") < 0, "Negative medicare payment"),
        (F.col("Tot_Dschrgs") < 0, "Negative discharges"),
        (F.col("Avg_Tot_Pymt_Amt") > F.col("Avg_Submtd_Cvrd_Chrg"), "Payment exceeds charge"),
        (F.col("Rndrng_Prvdr_CCN").isNull(), "Missing provider CCN"),
        (F.col("Rndrng_Prvdr_State_Abrvtn").isNull(), "Missing state")
    ]
    
    validation_notes_expr = F.concat_ws(
        " | ",
        *[F.when(condition, reason) for condition, reason in validation_conditions]
    )
    
    hospital_t4 = hospital_t3.withColumn(
        "VALIDATION_NOTES",
        validation_notes_expr
    ).withColumn(
        "DATA_QUALITY_FLAG",
        F.when(
            (F.col("VALIDATION_NOTES").isNull()) | (F.col("VALIDATION_NOTES") == ""),
            DATA_QUALITY_PASS
        ).otherwise("FAIL")
    )
    
    fail_count = hospital_t4.filter(F.col("DATA_QUALITY_FLAG") == "FAIL").count()
    pass_count = hospital_t4.filter(F.col("DATA_QUALITY_FLAG") == DATA_QUALITY_PASS).count()
    
    print(f"✓ H4 Complete: Data validation applied")
    print(f"  Passed validation: {pass_count:,}")
    print(f"  Failed validation: {fail_count:,}")
except Exception as e:
    print(f"✗ H4 Failed: {str(e)}")
    raise

## Write Hospital Silver Table
Save transformed hospital data to silver layer

In [0]:
try:
    print("Writing hospital data to Silver layer...\n")
    
    target_table = SILVER_HOSPITAL_TABLE
    
    hospital_t4.write.format("delta") \
        .mode(SILVER_WRITE_MODE) \
        .option("overwriteSchema", OVERWRITE_SCHEMA) \
        .saveAsTable(target_table)
    
    total_records = hospital_t4.count()
    
    print(f"✓ Hospital Silver table written successfully: {target_table}")
    print(f"  Total records: {total_records:,}")
    print(f"  Mode: overwrite")
    print(f"  Schema evolution: enabled\n")
except Exception as e:
    print(f"✗ Failed to write Hospital Silver table: {str(e)}")
    raise

## Hospital Silver Verification
Verify hospital silver table and display sample

In [0]:
try:
    hospital_silver = spark.table(SILVER_HOSPITAL_TABLE)
    
    total_records = hospital_silver.count()
    print(f"Hospital Silver Total Records: {total_records:,}\n")
    
    print("Sample of 10 records:\n")
    
    display_cols = [
        "Rndrng_Prvdr_CCN", "Rndrng_Prvdr_Org_Name", "Rndrng_Prvdr_City", 
        "Rndrng_Prvdr_State_Abrvtn", "DRG_Cd", "DRG_Desc", "Tot_Dschrgs",
        "Avg_Mdcr_Pymt_Amt", "LOCATION_TYPE", "DATA_QUALITY_FLAG"
    ]
    
    display(hospital_silver.select(display_cols).limit(10))
    
    print("\n" + "="*80)
    print("✓ Hospital Silver Layer Complete!")
    print("="*80)
except Exception as e:
    print(f"✗ Verification failed: {str(e)}")
    raise

In [0]:
silver_end = datetime.now()
silver_duration = (silver_end - silver_start).seconds

silver_output = spark.table(SILVER_CLAIMS_TABLE).count()

pass_count = spark.table(SILVER_CLAIMS_TABLE).filter(f"DATA_QUALITY_FLAG = '{DATA_QUALITY_PASS}'").count()

fail_count = silver_input - pass_count

print(f"{'='*50}")
print(f"SILVER AUDIT SUMMARY")
print(f"{'='*50}")
print(f"Input Records  : {silver_input:,}")
print(f"PASS Records   : {pass_count:,}")
print(f"FAIL Records   : {fail_count:,}")
print(f"Output Records : {silver_output:,}")
print(f"Start Time     : {silver_start}")
print(f"End Time       : {silver_end}")
print(f"Duration       : {silver_duration} seconds")
print(f"Status         : ✅ SUCCESS")
print(f"{'='*50}")